<img src="https://raw.githubusercontent.com/OmarLi-360/powerbi-tableau-analista/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 3 · Modelo estrella y relaciones
**Analista de datos · Power BI + Tableau** — Verificar el modelo antes de construir el informe · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/03_modelo_estrella.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/powerbi-tableau-analista/#clave-03)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Ver por qué conviene separar hechos y dimensiones.
2. Comprobar que cada llave sea única y que no haya ventas huérfanas.
3. Reconocer el síntoma de una relación que falta.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/powerbi-tableau-analista"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

In [ ]:
# @title Preparar el modelo: las tablas tal como quedan en Power BI
calendario = pd.DataFrame({"Date": pd.date_range("2024-01-01", "2026-12-31")})
calendario["Año"], calendario["Mes"] = calendario.Date.dt.year, calendario.Date.dt.month
metas = metas.assign(fecha_mes=pd.to_datetime(dict(year=metas.anio, month=metas.mes, day=1)))
TABLAS = {"Ventas": (ventas, "Hechos"), "Metas": (metas, "Hechos"), "Clientes": (clientes, "Dimensión"),
          "Productos": (productos, "Dimensión"), "Vendedores": (vendedores, "Dimensión"),
          "Sedes": (sedes, "Dimensión"), "Calendario": (calendario, "Dimensión")}
RELACIONES = [("Ventas", "id_cliente", "Clientes", "id_cliente"), ("Ventas", "sku", "Productos", "sku"),
              ("Ventas", "id_vendedor", "Vendedores", "id_vendedor"), ("Ventas", "id_sede", "Sedes", "id_sede"),
              ("Ventas", "fecha", "Calendario", "Date"), ("Metas", "id_sede", "Sedes", "id_sede"),
              ("Metas", "fecha_mes", "Calendario", "Date")]
display(Markdown(f"Modelo listo: **{len(TABLAS)} tablas** y **{len(RELACIONES)} relaciones**."))

## 1 · Hechos y dimensiones
En Power BI no se unen las tablas: se relacionan. La tabla de **hechos** (Ventas) tiene muchas filas y las **dimensiones** (clientes, productos, sedes…) pocas. Separarlas evita repetir el nombre del cliente, su segmento y su ciudad en cada una de las 17.106 líneas.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué tan grande es cada tabla del modelo? |
> | **Qué se grafica** | Barras horizontales: filas por tabla, en escala logarítmica · Azul: hechos · Gris: dimensiones |
> | **Por qué esta forma** | Las tablas van de 6 a 17.106 filas: en escala normal las dimensiones no se verían. |
> | **Qué se descartó** | Una tabla de conteos: no muestra la diferencia de tamaño de un vistazo. |
> | **Cómo se lee** | Las tablas grandes (hechos) quedan al centro de la estrella; las pequeñas (dimensiones) alrededor. |
> | **En Excel 365** | Vista de tabla de Power BI: el contador de filas está abajo a la izquierda. |

In [ ]:
# @title Gráfico 1 · Filas por tabla
filas = pd.Series({n: len(t) for n, (t, _) in TABLAS.items()}).sort_values()
colores = [AZUL if TABLAS[n][1] == "Hechos" else GRIS for n in filas.index]
fig = go.Figure(go.Bar(x=filas.values, y=[f"{n} ({TABLAS[n][1].lower()})" for n in filas.index], orientation="h",
                       marker_color=colores, text=[miles(v) for v in filas.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"Ventas tiene {miles(filas['Ventas'])} filas; la dimensión más pequeña, {filas.min()}",
                  xaxis_type="log", xaxis_title="Filas (escala logarítmica)", height=360)
fig.show()

## 2 · Las llaves: únicas en la dimensión y sin huérfanos en los hechos

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cada relación cumple las dos condiciones de una llave? |
> | **Qué se grafica** | Tabla de verificación: valores repetidos en la dimensión y valores sin pareja en los hechos, por relación |
> | **Por qué esta forma** | Son comprobaciones de sí o no: una tabla con ✓ es más clara que un gráfico. |
> | **Qué se descartó** | Confiar en que Power BI avisará: si la llave se repite, Power BI crea una relación varios a varios sin preguntar. |
> | **Cómo se lee** | Las dos columnas deben estar en cero. Un repetido o un huérfano produce totales que no cuadran. |
> | **En Excel 365** | En Power Query: selecciona la llave › Inicio › Mantener filas › Mantener duplicados. Si queda vacía, la llave es única. |

In [ ]:
# @title Verificar las llaves de cada relación
filas_v = []
for hecho, lh, dim, ld in RELACIONES:
    th, td = TABLAS[hecho][0], TABLAS[dim][0]
    llaves_h = th[lh].drop_duplicates()
    if ld == "Date":
        llaves_h = pd.to_datetime(llaves_h)
    repetidos = int(td[ld].duplicated().sum())
    huerfanos = int((~llaves_h.isin(td[ld])).sum())
    filas_v.append({"Relación": f"{hecho}[{lh}] → {dim}[{ld}]", "Repetidos en la dimensión": repetidos,
                    "Sin pareja en los hechos": huerfanos, "Estado": "✓" if repetidos == huerfanos == 0 else "✗ revisar"})
verif = pd.DataFrame(filas_v)
display(verif.style.hide(axis="index"))
display(Markdown(f"**{(verif.Estado == '✓').sum()} de {len(verif)}** relaciones cumplen las dos condiciones."))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Por qué la relación es varios a uno? |
> | **Qué se grafica** | Histograma: cuántas líneas de venta tiene cada cliente |
> | **Por qué esta forma** | Muestra que un cliente aparece muchas veces en Ventas (el lado «varios») y una sola vez en Clientes (el lado «uno»). |
> | **Qué se descartó** | Un promedio: escondería que hay clientes con 3 líneas y otros con más de 100. |
> | **Cómo se lee** | Cada barra agrupa clientes según su número de líneas. Ninguno tiene una sola: por eso es *:1 y no 1:1. |
> | **En Excel 365** | Vista de modelo: doble clic en la relación Ventas–Clientes y revisa «Cardinalidad: Varios a uno (*:1)». |

In [ ]:
# @title Gráfico 2 · Líneas de venta por cliente
por_cliente = ventas.groupby("id_cliente").size()
fig = go.Figure(go.Histogram(x=por_cliente, nbinsx=30, marker_color=AZUL, marker_line_color="white", marker_line_width=1))
fig.update_layout(title=f"Cada cliente aparece en promedio {por_cliente.mean():.0f} veces en Ventas y una sola vez en Clientes".replace(".", ","),
                  xaxis_title="Líneas de venta del cliente", yaxis_title="Clientes", height=340)
fig.show()

## 3 · El síntoma de una relación que falta

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué muestra la matriz si falta la relación Ventas–Clientes? |
> | **Qué se grafica** | Barras: venta neta 2025 por segmento, con y sin la relación |
> | **Por qué esta forma** | Compara el resultado correcto con el síntoma, segmento por segmento. |
> | **Qué se descartó** | Solo explicar el error: hay que verlo para reconocerlo en un tablero real. |
> | **Cómo se lee** | Sin relación, cada segmento muestra el total de la empresa. Con la relación, cada uno muestra su parte. |
> | **En Excel 365** | Matriz con Clientes[segmento] en filas y [Venta neta] en valores. |

In [ ]:
# @title Gráfico 3 · Venta por segmento con y sin relación
relacion = "Sin relación"  # @param ["Activa", "Sin relación"]
v25 = df[(df.anio == 2025) & (df.estado != "Anulada")]
total = v25.subtotal.sum()
seg = v25.groupby("segmento")["subtotal"].sum() if relacion == "Activa" else pd.Series(total, index=sorted(clientes.segmento.unique()))
seg = seg.sort_values() / 1e6
fig = go.Figure(go.Bar(x=seg.values, y=seg.index, orientation="h", marker_color=AZUL if relacion == "Activa" else MAGENTA,
                       text=[f"{v:,.0f}".replace(",", ".") for v in seg.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=("Cada segmento muestra su venta y la suma da el total" if relacion == "Activa"
                         else f"Sin relación, los {len(seg)} segmentos muestran el total: {millones(total)}"),
                  xaxis_title=MM, height=320)
dar_aire(fig, seg.max())
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Cuántas filas tiene la tabla Calendario de 2024 a 2026?
respuesta = "Elige una opción"  # @param ["Elige una opción", "365", "731", "1.096", "36"]
revisar(respuesta, miles(len(calendario)), """Una fila por día: 366 de 2024 (bisiesto) + 365 + 365.""")

In [ ]:
# @title Reto 2 · ¿Cuál es la cardinalidad de la relación Ventas–Productos?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Uno a uno (1:1)", "Varios a uno (*:1)", "Varios a varios (*:*)", "Uno a varios desde Ventas"]
revisar(respuesta, "Varios a uno (*:1)", """Muchas líneas de venta son del mismo producto; cada producto aparece una vez en Productos.""")

In [ ]:
# @title Reto 3 · Una matriz muestra el mismo total en todas las filas. ¿Qué revisas primero?
respuesta = "Elige una opción"  # @param ["Elige una opción", "El formato del número", "La relación entre la tabla de las filas y Ventas", "El tipo de visual", "El tamaño de la página"]
revisar(respuesta, "La relación entre la tabla de las filas y Ventas", """Es el síntoma del gráfico 3: el filtro de la fila no llega a los hechos.""")

## Apóyate en IA
Para revisar un modelo con ayuda de IA, describe las tablas y relaciones sin pegar datos:

```text
Tengo en Power BI: Ventas (hechos), Metas (hechos) y las dimensiones Clientes, Productos, Vendedores, Sedes y Calendario.
Relaciones: [LISTA: tabla[campo] → tabla[campo], cardinalidad, dirección].
¿Es un modelo estrella correcto? ¿Qué relación falta o sobra y qué síntoma produciría?
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* En un modelo estrella, los hechos van al centro y las dimensiones alrededor.
* Cada llave debe ser única en la dimensión y no dejar huérfanos en los hechos.
* El mismo total en todas las filas es el síntoma de una relación que falta.

**Siguiente:** [Clave 4 · Medidas DAX](https://omarli-360.github.io/powerbi-tableau-analista/#clave-04) · [Cuaderno 04](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/04_medidas_dax.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>